In [3]:
# Standardize dataset column names

data = df.copy()

column_mapping = {
    "customerid": "CustomerId",
    "surname": "Surname",
    "creditscore": "CreditScore",
    "geography": "Geography",
    "gender": "Gender",
    "age": "Age",
    "tenure": "Tenure",
    "balance": "Balance",
    "numofproducts": "NumOfProducts",
    "hascrcard": "HasCrCard",
    "isactivemember": "IsActiveMember",
    "estimatedsalary": "EstimatedSalary",
    "exited": "Exited",
    "year": "Year"
}

data.rename(columns=column_mapping, inplace=True)

required_columns = [
    "CustomerId", "CreditScore", "Geography", "Gender",
    "Age", "Tenure", "Balance", "NumOfProducts",
    "HasCrCard", "IsActiveMember", "EstimatedSalary", "Exited"
]

# IMPORTANT: Validate data.columns, NOT df.columns
missing_columns = [
    col for col in required_columns
    if col not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Columns still missing: {missing_columns}"
    )

print("SUCCESS: All required columns are available.")
print("Dataset shape:", data.shape)
print("Validated columns:", required_columns)

display(data[required_columns].head())

SUCCESS: All required columns are available.
Dataset shape: (10000, 26)
Validated columns: ['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:

# Generate Research Paper, Executive Summary, and Supporting Analysis Tables

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, mannwhitneyu

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)
from xml.sax.saxutils import escape

warnings.filterwarnings("ignore")


# ---------------------------------------------------------
# Load the existing project dataset
# ---------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy"
)

RAW_FILE = PROJECT_DIR / "data" / "raw" / "European_Bank.csv"

PROCESSED_FILE = (
    PROJECT_DIR / "data" / "processed" /
    "European_Bank_Cleaned_Analytical.csv"
)

REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if PROCESSED_FILE.exists():
    DATA_FILE = PROCESSED_FILE
elif RAW_FILE.exists():
    DATA_FILE = RAW_FILE
else:
    raise FileNotFoundError(
        "European_Bank.csv was not found in the project data folders."
    )

df = pd.read_csv(
    DATA_FILE,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

df.columns = (
    df.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
    .str.lower()
)

print("Dataset loaded successfully.")
print("Dataset file:", DATA_FILE)
print("Dataset shape:", df.shape)


# ---------------------------------------------------------
# Standardize column names
# ---------------------------------------------------------

column_mapping = {
    "customerid": "CustomerId",
    "surname": "Surname",
    "creditscore": "CreditScore",
    "geography": "Geography",
    "gender": "Gender",
    "age": "Age",
    "tenure": "Tenure",
    "balance": "Balance",
    "numofproducts": "NumOfProducts",
    "hascrcard": "HasCrCard",
    "isactivemember": "IsActiveMember",
    "estimatedsalary": "EstimatedSalary",
    "exited": "Exited",
    "year": "Year",
    "age_group": "Age_Group",
    "tenure_group": "Tenure_Group",
    "balance_category": "Balance_Category",
    "product_count_category": "Product_Count_Category",
    "activity_status": "Activity_Status",
    "customer_status": "Customer_Status",
    "high_balance_flag": "High_Balance_Flag",
    "inactive_high_balance_flag": "Inactive_High_Balance_Flag",
    "product_relationship": "Product_Relationship",
    "premium_customer_candidate": "Premium_Customer_Candidate",
    "relationship_strength_index": "Relationship_Strength_Index",
    "relationship_strength_category": "Relationship_Strength_Category"
}

data = df.rename(columns=column_mapping).copy()

required_columns = [
    "CustomerId",
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

missing_columns = [
    col for col in required_columns
    if col not in data.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print("All required columns are available.")


# ---------------------------------------------------------
# Prepare analytical data
# ---------------------------------------------------------

numeric_columns = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
]

for column in numeric_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="coerce"
    )

data = data.dropna(
    subset=[
        "Exited",
        "IsActiveMember",
        "NumOfProducts"
    ]
).copy()

data["Exited"] = data["Exited"].astype(int)
data["IsActiveMember"] = data["IsActiveMember"].astype(int)

if not set(data["Exited"].unique()).issubset({0, 1}):
    raise ValueError("Exited must contain only 0 and 1.")

if not set(data["IsActiveMember"].unique()).issubset({0, 1}):
    raise ValueError("IsActiveMember must contain only 0 and 1.")

total = len(data)

if total == 0:
    raise ValueError("No usable customer records were found.")

exited = int(data["Exited"].sum())
retained = total - exited

churn_rate = exited / total * 100
retention_rate = retained / total * 100


# ---------------------------------------------------------
# Overall customer retention and engagement analysis
# ---------------------------------------------------------

active = data[data["IsActiveMember"] == 1]
inactive = data[data["IsActiveMember"] == 0]

active_churn = (
    active["Exited"].mean() * 100
    if len(active) else np.nan
)

inactive_churn = (
    inactive["Exited"].mean() * 100
    if len(inactive) else np.nan
)

active_retention = 100 - active_churn
inactive_retention = 100 - inactive_churn

err = (
    active_retention / inactive_retention
    if pd.notna(inactive_retention) and inactive_retention != 0
    else np.nan
)


# ---------------------------------------------------------
# Product adoption analysis
# ---------------------------------------------------------

pdi = data["NumOfProducts"].mean()

product_churn = (
    data.groupby("NumOfProducts")["Exited"]
    .agg(
        Customers="count",
        Exited="sum",
        Churn_Rate="mean"
    )
    .reset_index()
)

product_churn["Churn_Rate_Percent"] = (
    product_churn["Churn_Rate"] * 100
)


# ---------------------------------------------------------
# Customer demographic and geographic analysis
# ---------------------------------------------------------

def group_churn(column):

    result = (
        data.groupby(column, dropna=False)["Exited"]
        .agg(
            Customers="count",
            Exited="sum",
            Churn_Rate="mean"
        )
        .reset_index()
    )

    result["Churn_Rate"] = (
        result["Churn_Rate"] * 100
    )

    return result


geography_results = group_churn("Geography")
gender_results = group_churn("Gender")

data["Age_Group"] = pd.cut(
    data["Age"],
    bins=[0, 25, 35, 45, 55, 100],
    labels=[
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56+"
    ]
)

age_results = group_churn("Age_Group")
tenure_results = group_churn("Tenure")


# ---------------------------------------------------------
# Financial and high-balance customer risk analysis
# ---------------------------------------------------------

balance_threshold = data["Balance"].quantile(0.75)

high_balance = data[
    data["Balance"] >= balance_threshold
]

inactive_high_balance = high_balance[
    high_balance["IsActiveMember"] == 0
]

hbdr = (
    len(inactive_high_balance) / len(high_balance) * 100
    if len(high_balance) else np.nan
)

inactive_high_balance_churn = (
    inactive_high_balance["Exited"].mean() * 100
    if len(inactive_high_balance) else np.nan
)


# ---------------------------------------------------------
# Credit card retention analysis
# ---------------------------------------------------------

card = data[data["HasCrCard"] == 1]
no_card = data[data["HasCrCard"] == 0]

card_retention = (
    (1 - card["Exited"].mean()) * 100
    if len(card) else np.nan
)

no_card_retention = (
    (1 - no_card["Exited"].mean()) * 100
    if len(no_card) else np.nan
)

ccs = card_retention - no_card_retention


# ---------------------------------------------------------
# Statistical validation
# ---------------------------------------------------------

statistical_results = []


def add_chi_square(variable):

    table = pd.crosstab(
        data[variable],
        data["Exited"]
    )

    if table.shape[0] >= 2 and table.shape[1] >= 2:

        chi2, p_value, dof, expected = chi2_contingency(table)

        n = table.to_numpy().sum()

        denominator = min(
            table.shape[0] - 1,
            table.shape[1] - 1
        )

        cramers_v = (
            np.sqrt(chi2 / (n * denominator))
            if denominator > 0 and n > 0
            else np.nan
        )

        statistical_results.append({
            "Variable": variable,
            "Test": "Chi-square independence",
            "Statistic": chi2,
            "P_Value": p_value,
            "Effect_Size": cramers_v,
            "Effect_Size_Name": "Cramer's V"
        })


def add_mann_whitney(variable):

    exited_values = data.loc[
        data["Exited"] == 1,
        variable
    ].dropna()

    retained_values = data.loc[
        data["Exited"] == 0,
        variable
    ].dropna()

    if len(exited_values) > 0 and len(retained_values) > 0:

        statistic, p_value = mannwhitneyu(
            exited_values,
            retained_values,
            alternative="two-sided"
        )

        statistical_results.append({
            "Variable": variable,
            "Test": "Mann-Whitney U",
            "Statistic": statistic,
            "P_Value": p_value,
            "Effect_Size": np.nan,
            "Effect_Size_Name": "Not calculated"
        })


for variable in [
    "Geography",
    "Gender",
    "IsActiveMember",
    "NumOfProducts",
    "HasCrCard"
]:
    add_chi_square(variable)


for variable in [
    "Age",
    "CreditScore",
    "Tenure",
    "Balance",
    "EstimatedSalary"
]:
    add_mann_whitney(variable)


stats_df = pd.DataFrame(statistical_results)


# ---------------------------------------------------------
# Develop retention KPI results
# ---------------------------------------------------------

kpi_df = pd.DataFrame([
    ["Total Customers", total, "Customers"],
    ["Exited Customers", exited, "Customers"],
    ["Retained Customers", retained, "Customers"],
    ["Customer Churn Rate", churn_rate, "%"],
    ["Customer Retention Rate", retention_rate, "%"],
    ["Active Member Churn Rate", active_churn, "%"],
    ["Inactive Member Churn Rate", inactive_churn, "%"],
    ["Engagement Retention Ratio (ERR)", err, "Ratio"],
    ["Product Diversity Index (PDI)", pdi, "Average products"],
    ["High-Balance Dormancy Rate (HBDR)", hbdr, "%"],
    ["Cardholder Retention", card_retention, "%"],
    ["Non-Cardholder Retention", no_card_retention, "%"],
    ["Cardholder Churn Spread (CCS)", ccs, "Percentage points"]
], columns=["KPI", "Value", "Unit"])


# ---------------------------------------------------------
# Configure PDF report formatting
# ---------------------------------------------------------

styles = getSampleStyleSheet()

styles.add(
    ParagraphStyle(
        name="ReportTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=18,
        leading=23,
        spaceAfter=14
    )
)

styles.add(
    ParagraphStyle(
        name="SectionHeading",
        parent=styles["Heading2"],
        fontSize=12,
        leading=15,
        spaceBefore=11,
        spaceAfter=6
    )
)

styles.add(
    ParagraphStyle(
        name="ReportBody",
        parent=styles["BodyText"],
        fontSize=9,
        leading=13,
        spaceAfter=7
    )
)

styles.add(
    ParagraphStyle(
        name="SmallText",
        parent=styles["BodyText"],
        fontSize=7,
        leading=9
    )
)


def para(text, style="ReportBody"):

    if pd.isna(text):
        text = "Not available"

    return Paragraph(
        escape(str(text)),
        styles[style]
    )


def add_section(story, heading, body):

    story.append(
        para(heading, "SectionHeading")
    )

    if isinstance(body, (list, tuple)):

        for item in body:
            story.append(para(item))

    else:
        story.append(para(body))


def add_table(story, frame, max_rows=35):

    if frame is None or frame.empty:

        story.append(
            para("No results were available for this section.")
        )

        return

    view = frame.head(max_rows).copy()

    rows = [
        [
            para(column, "SmallText")
            for column in view.columns
        ]
    ]

    for record in view.itertuples(
        index=False,
        name=None
    ):

        cells = []

        for value in record:

            if pd.isna(value):
                value = "Not available"

            elif isinstance(value, (float, np.floating)):
                value = f"{value:,.4f}"

            cells.append(
                para(value, "SmallText")
            )

        rows.append(cells)

    table = Table(
        rows,
        repeatRows=1
    )

    table.setStyle(
        TableStyle([
            (
                "BACKGROUND",
                (0, 0),
                (-1, 0),
                colors.HexColor("#17365D")
            ),
            (
                "TEXTCOLOR",
                (0, 0),
                (-1, 0),
                colors.white
            ),
            (
                "GRID",
                (0, 0),
                (-1, -1),
                0.35,
                colors.grey
            ),
            (
                "VALIGN",
                (0, 0),
                (-1, -1),
                "TOP"
            ),
            (
                "ROWBACKGROUNDS",
                (0, 1),
                (-1, -1),
                [
                    colors.white,
                    colors.HexColor("#EAF0F6")
                ]
            ),
            (
                "LEFTPADDING",
                (0, 0),
                (-1, -1),
                5
            ),
            (
                "RIGHTPADDING",
                (0, 0),
                (-1, -1),
                5
            ),
            (
                "TOPPADDING",
                (0, 0),
                (-1, -1),
                5
            ),
            (
                "BOTTOMPADDING",
                (0, 0),
                (-1, -1),
                5
            )
        ])
    )

    story.append(table)
    story.append(Spacer(1, 9))


def page_number(canvas, doc):

    canvas.saveState()

    canvas.setFont("Helvetica", 8)

    canvas.drawString(
        0.6 * inch,
        0.4 * inch,
        "Customer Engagement & Product Utilization Analytics"
    )

    canvas.drawRightString(
        7.9 * inch,
        0.4 * inch,
        f"Page {doc.page}"
    )

    canvas.restoreState()


# ---------------------------------------------------------
# Create the research paper
# ---------------------------------------------------------

paper = [
    para(
        "Customer Engagement & Product Utilization Analytics for Retention Strategy",
        "ReportTitle"
    ),
    para(
        "Research Paper | Customer Churn, Engagement, Product Adoption and Retention"
    ),
    Spacer(1, 12)
]

add_section(
    paper,
    "Abstract",
    f"This study analyses {total:,} customer records to examine "
    f"engagement, product utilization and customer churn. The observed "
    f"churn rate is {churn_rate:.2f}% ({exited:,} customers). "
    "Descriptive summaries and statistical association tests are used "
    "to identify customer groups for retention review. Findings are "
    "observational and do not establish causality."
)

add_section(
    paper,
    "1. Research Problem",
    "Customer attrition can reduce customer lifetime value and weaken "
    "long-term banking relationships. This study investigates customer "
    "engagement and product-utilization patterns associated with churn."
)

add_section(
    paper,
    "2. Project Introduction",
    "This project applies customer-level analytics to examine the "
    "relationship between activity status, product holdings, customer "
    "characteristics and exit status."
)

add_section(
    paper,
    "3. Business Background",
    "Banks operate in a competitive environment where retaining "
    "existing customers is an important relationship-management objective. "
    "Activity and product holdings can help identify groups for service "
    "and engagement reviews."
)

add_section(
    paper,
    "4. Research Objectives",
    [
        "Measure the overall customer churn and retention baseline.",
        "Compare churn between active and inactive customers.",
        "Evaluate churn patterns by product count.",
        "Examine geography, gender, age and tenure differences.",
        "Identify inactive high-balance customers for further review.",
        "Define retention KPIs and test selected statistical associations.",
        "Develop practical retention recommendations."
    ]
)

add_section(
    paper,
    "5. Dataset and Variables",
    f"The analysis uses {total:,} usable records from {DATA_FILE.name}. "
    "The original dataset includes customer identification, credit score, "
    "geography, gender, age, tenure, balance, product count, credit card "
    "status, activity status, estimated salary and exit status. "
    "Exited is the target outcome (1 = exited, 0 = retained)."
)

add_section(
    paper,
    "6. Data Preparation Methodology",
    "The project uses a separate cleaned analytical dataset while "
    "preserving the raw source. This report loads the existing cleaned "
    "dataset, standardizes column names in memory, converts analytical "
    "fields to numeric types and excludes records missing essential "
    "analysis fields. It does not overwrite either source dataset."
)

add_section(
    paper,
    "7. Feature Engineering",
    "Derived features include age bands and a high-balance segment "
    "based on the 75th percentile of Balance. Activity status and "
    "product count are used as engagement and utilization proxies, "
    "not direct measures of customer satisfaction."
)

add_section(
    paper,
    "8. Exploratory Data Analysis Findings",
    [
        f"Usable records: {total:,}.",
        f"Exited customers: {exited:,}.",
        f"Retained customers: {retained:,}.",
        f"Overall churn: {churn_rate:.2f}%.",
        f"Overall retention: {retention_rate:.2f}%."
    ]
)

add_section(paper, "Geographic Analysis", "Customer distribution and churn by geography.")
add_table(paper, geography_results)

add_section(paper, "Gender Analysis", "Customer distribution and churn by gender.")
add_table(paper, gender_results)

add_section(paper, "Age Analysis", "Customer distribution and churn by age group.")
add_table(paper, age_results)

add_section(paper, "Tenure Analysis", "Customer distribution and churn by tenure.")
add_table(paper, tenure_results)


# ---------------------------------------------------------
# Engagement findings
# ---------------------------------------------------------

add_section(
    paper,
    "9. Engagement Findings",
    [
        f"Active customers: {len(active):,}; churn: {active_churn:.2f}%.",
        f"Inactive customers: {len(inactive):,}; churn: {inactive_churn:.2f}%.",
        f"Engagement Retention Ratio (ERR): {err:.4f}.",
        "Observed churn differs between active and inactive customers. "
        "This association does not prove causation."
    ]
)


# ---------------------------------------------------------
# Product adoption findings
# ---------------------------------------------------------

add_section(
    paper,
    "10. Product Adoption Findings",
    "Product count is treated as a proxy for product utilization. "
    "Customer counts and churn rates are shown below."
)

add_table(
    paper,
    product_churn[
        [
            "NumOfProducts",
            "Customers",
            "Exited",
            "Churn_Rate_Percent"
        ]
    ]
)


# ---------------------------------------------------------
# Financial and premium customer risk findings
# ---------------------------------------------------------

add_section(
    paper,
    "11. Financial and Premium Customer Risk",
    f"The high-balance threshold is {balance_threshold:,.2f}, "
    f"representing the 75th percentile of Balance. The segment "
    f"contains {len(high_balance):,} customers; "
    f"{len(inactive_high_balance):,} are inactive. "
    f"HBDR is {hbdr:.2f}%. Churn within inactive high-balance "
    f"customers is {inactive_high_balance_churn:.2f}%. "
    "This is a screening segment, not a verified premium-customer classification."
)


# ---------------------------------------------------------
# Retention KPI definitions and results
# ---------------------------------------------------------

add_section(
    paper,
    "12. KPI Definitions and Results",
    "ERR = active-customer retention divided by inactive-customer "
    "retention. PDI = average number of products held. HBDR = inactive "
    "customers among high-balance customers. CCS = cardholder retention "
    "minus non-cardholder retention, in percentage points."
)

add_table(paper, kpi_df)


# ---------------------------------------------------------
# Statistical validation findings
# ---------------------------------------------------------

add_section(
    paper,
    "13. Statistical Validation",
    "Chi-square tests evaluate associations between categorical "
    "variables and Exited; Cramer's V is reported as an effect size. "
    "Mann-Whitney U tests compare numeric-variable distributions "
    "between exited and retained groups. Tests are two-sided. "
    "A p-value below 0.05 indicates statistical evidence of association, "
    "not causation. Effect size and business relevance should also be considered."
)

add_table(paper, stats_df)


# ---------------------------------------------------------
# Business recommendations
# ---------------------------------------------------------

add_section(
    paper,
    "14. Business Recommendations",
    [
        "Prioritize relationship reviews for inactive customers.",
        "Investigate high-churn product-count groups and validate group sizes before intervention.",
        "Review inactive high-balance customers through a service-led retention process.",
        "Use suitability and customer-needs checks before cross-selling products.",
        "Monitor churn, retention and engagement KPIs over time."
    ]
)


# ---------------------------------------------------------
# Research limitations
# ---------------------------------------------------------

add_section(
    paper,
    "15. Limitations",
    [
        "The dataset is a customer snapshot and does not establish the timing or causes of exit.",
        "Activity and product count are proxies; satisfaction and interaction history are not directly measured.",
        "Observational associations do not prove causation.",
        "The analysis does not measure the effectiveness of retention interventions.",
        "The high-balance threshold is a statistical rule, not a formal premium-customer definition.",
        "Results may not generalize to other banks, years or populations."
    ]
)


# ---------------------------------------------------------
# Research conclusion and references
# ---------------------------------------------------------

add_section(
    paper,
    "16. Conclusion",
    f"The observed churn rate is {churn_rate:.2f}%. Activity status "
    "and product count show differences in churn across customer groups. "
    "These findings provide a baseline for targeted customer review and "
    "retention measurement. Future work should incorporate longitudinal "
    "behaviour and intervention outcomes."
)

add_section(
    paper,
    "17. References",
    [
        "European Bank customer dataset supplied for this project.",
        "Pandas documentation: https://pandas.pydata.org/docs/.",
        "SciPy documentation: https://docs.scipy.org/doc/scipy/.",
        "ReportLab documentation: https://www.reportlab.com/docs/reportlab-userguide.pdf."
    ]
)

add_section(
    paper,
    "18. Reproducibility",
    f"Source file: {DATA_FILE}. "
    f"Report generated: {pd.Timestamp.today().date()}. "
    "Results are based on the records successfully loaded in this run."
)


# ---------------------------------------------------------
# Export the research paper PDF
# ---------------------------------------------------------

paper_pdf = (
    REPORT_DIR /
    "Customer_Engagement_Research_Paper.pdf"
)

paper_document = SimpleDocTemplate(
    str(paper_pdf),
    pagesize=A4,
    rightMargin=0.6 * inch,
    leftMargin=0.6 * inch,
    topMargin=0.65 * inch,
    bottomMargin=0.7 * inch
)

paper_document.build(
    paper,
    onFirstPage=page_number,
    onLaterPages=page_number
)


# ---------------------------------------------------------
# Create the executive summary PDF
# ---------------------------------------------------------

summary = [
    para("Executive Summary", "ReportTitle"),
    para(
        "Customer Engagement & Product Utilization Analytics for Retention Strategy"
    ),
    Spacer(1, 10)
]

add_section(
    summary,
    "Business Challenge",
    "Customer churn affects customer relationships and portfolio "
    "performance. This project identifies groups with different observed "
    "churn patterns to support retention planning."
)

add_section(
    summary,
    "Dataset Snapshot",
    f"The analysis includes {total:,} customer records with demographic, "
    "geographic, tenure, financial, product, activity and exit-status fields."
)

summary_kpis = pd.DataFrame([
    ["Total customers", f"{total:,}"],
    ["Exited customers", f"{exited:,}"],
    ["Retained customers", f"{retained:,}"],
    ["Overall churn", f"{churn_rate:.2f}%"],
    ["Overall retention", f"{retention_rate:.2f}%"],
    ["Active-member churn", f"{active_churn:.2f}%"],
    ["Inactive-member churn", f"{inactive_churn:.2f}%"],
    ["Average products per customer", f"{pdi:.2f}"],
    ["Inactive share of high-balance group", f"{hbdr:.2f}%"]
], columns=["Measure", "Result"])

add_table(summary, summary_kpis)

add_section(
    summary,
    "Key Findings",
    [
        f"Inactive-member churn is {inactive_churn:.2f}%, "
        f"compared with {active_churn:.2f}% for active members.",
        "Product-count groups show different observed churn rates.",
        f"The high-balance segment contains {len(high_balance):,} customers, "
        f"including {len(inactive_high_balance):,} inactive customers.",
        f"Cardholder retention is {card_retention:.2f}%; "
        f"non-cardholder retention is {no_card_retention:.2f}%."
    ]
)

add_section(
    summary,
    "Recommended Actions",
    [
        "Prioritize engagement and relationship reviews for inactive customers.",
        "Investigate high-churn product-count groups before intervention.",
        "Review inactive high-balance customers through a service-led process.",
        "Apply customer-needs and suitability checks before cross-selling.",
        "Track retention KPIs consistently and evaluate future outcomes."
    ]
)

add_section(
    summary,
    "Important Cautions",
    "The analysis is observational and identifies associations, not causes. "
    "Activity and product count are proxies. The high-balance group is not "
    "a verified premium-customer classification."
)

add_section(
    summary,
    "Conclusion",
    "The project provides a measurable baseline for retention planning. "
    "Future evaluation should determine whether targeted engagement actions improve retention."
)


# ---------------------------------------------------------
# Export the executive summary PDF
# ---------------------------------------------------------

summary_pdf = (
    REPORT_DIR /
    "Customer_Engagement_Executive_Summary.pdf"
)

summary_document = SimpleDocTemplate(
    str(summary_pdf),
    pagesize=A4,
    rightMargin=0.6 * inch,
    leftMargin=0.6 * inch,
    topMargin=0.65 * inch,
    bottomMargin=0.7 * inch
)

summary_document.build(
    summary,
    onFirstPage=page_number,
    onLaterPages=page_number
)


# ---------------------------------------------------------
# Export supporting analysis tables
# ---------------------------------------------------------

kpi_csv = REPORT_DIR / "research_paper_kpi_results.csv"

stats_csv = REPORT_DIR / "research_paper_statistical_tests.csv"

product_csv = REPORT_DIR / "research_paper_product_churn.csv"

kpi_df.to_csv(kpi_csv, index=False)
stats_df.to_csv(stats_csv, index=False)
product_churn.to_csv(product_csv, index=False)


# ---------------------------------------------------------
# Verify all generated deliverables
# ---------------------------------------------------------

output_files = [
    paper_pdf,
    summary_pdf,
    kpi_csv,
    stats_csv,
    product_csv
]

for output_file in output_files:

    if not output_file.exists():
        raise FileNotFoundError(
            f"Output file was not generated: {output_file}"
        )

    if output_file.stat().st_size == 0:
        raise RuntimeError(
            f"Output file is empty: {output_file}"
        )


# ---------------------------------------------------------
# Display final Phase 10 execution results
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("PHASE 10 — RESEARCH PAPER & DOCUMENTATION")
print("=" * 65)

print("\nDATASET INFORMATION")
print("Dataset:", DATA_FILE)
print("Records analysed:", f"{total:,}")
print("Columns in source dataset:", len(df.columns))

print("\nKEY RESULTS")
print("Overall churn:", f"{churn_rate:.2f}%")
print("Overall retention:", f"{retention_rate:.2f}%")
print("Active-member churn:", f"{active_churn:.2f}%")
print("Inactive-member churn:", f"{inactive_churn:.2f}%")
print("Average products:", f"{pdi:.2f}%")
print("Statistical tests performed:", len(stats_df))

print("\nGENERATED PDF REPORTS")

print(paper_pdf)
print(summary_pdf)

print("\nGENERATED SUPPORTING CSV FILES")

print(kpi_csv)
print(stats_csv)
print(product_csv)

print("\n" + "=" * 65)
print("SUCCESS: ALL FIVE DELIVERABLES GENERATED AND VERIFIED.")
print("=" * 65)


Dataset loaded successfully.
Dataset file: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\European_Bank_Cleaned_Analytical.csv
Dataset shape: (10000, 26)
All required columns are available.

PHASE 10 — RESEARCH PAPER & DOCUMENTATION

DATASET INFORMATION
Dataset: C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\European_Bank_Cleaned_Analytical.csv
Records analysed: 10,000
Columns in source dataset: 26

KEY RESULTS
Overall churn: 20.37%
Overall retention: 79.63%
Active-member churn: 14.27%
Inactive-member churn: 26.85%
Average products: 1.53%
Statistical tests performed: 10

GENERATED PDF REPORTS
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\reports\Customer_Engagement_Research_Paper.pdf
C:\Users\hp\Customer Engagement & Product Utilization Analytics for Retention Strategy\reports\Customer_Engagement_Executive_Summary.pdf

GENERATED SUPPORTI